# T11 It2: pre-registration, then the fresh NUDT 909 draw scored once

Part 1 writes `results/preregistered_t11.json` and stops if it already exists. It was run before the 909 dump finished, and nothing reads 909
before part 2. Judges are trained on the 378 satellite-made photos only; their cuts are set on the NUDT design sets (101-606); nothing is
tuned on official TEST / NUDT 707 / 808 / 909.

| id | rule | role |
|---|---|---|
| base | bar 8 / 11 (PC); bar 8, cap 10, M = 5 (Jetson today) | reference |
| S5z | S5 (relative inliers + scale augmentation, satellite-trained), zero-wrong cut (0.9995 PC / 0.999 Jetson), no override; Jetson early exit at max(0.98, cut) | **primary** (0 wrong on every design set and official TRY) |
| S1z | S1 (T7 architecture, satellite-trained), zero-wrong cut 0.9995 | secondary |
| S5l, S1l | the same judges at the Learn-then-Test cut 0.998 (Jetson S5: 0.995): 1-3 wrong on design sets | secondary (does not meet the 0-wrong rule) |
| noov | today's bars without any override (today's rule has none: identical to base) | — |

Success (primary): on NUDT 909, 0 wrong and no more wrong than the base; gain >= 0; a paired-bootstrap interval above 0 counts as a score gain,
otherwise "safety only". Robustness: inlier signals x0.5 ... x2 on 909.

In [1]:
import os, sys, json, hashlib, datetime, gc
os.environ.setdefault("OMP_NUM_THREADS", "4")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
import train.checks.candidate_judge as cj
import train.checks.scale_robust as sr
from train.checks.decision_signals import build_signal_table
from train.checks.holdout import NUDT_DESIGN, answer_change, bar_frontier, versus_frontier
from train.checks.eval_tools import failure_log, manifest, save_manifest
R = ROOT / "tests/Test_Iteration_11/results"
PRE_PATH = R / "preregistered_t11.json"
if not PRE_PATH.exists():
    PRE = dict(written=datetime.datetime.now().isoformat(timespec="seconds"), final_set="nudt909",
               note="NUDT fresh 909: 300 photos, pool 2400, the 500-pool and every 404/505/606/707/808 photo excluded; dumped before any T11 rule was "
                    "chosen; read only below. Judges trained on 378 satellite-made photos (Cartosat scene B, Pleiades Neo; T11 Sat_Photo_Dump); cuts on NUDT 101-606.",
               finalists={"pc": {"S5z": dict(judge="S5", cut=0.9995), "S1z": dict(judge="S1", cut=0.9995), "S5l": dict(judge="S5", cut=0.998), "S1l": dict(judge="S1", cut=0.998)},
                          "jetson": {"S5z": dict(judge="S5", cut=0.999), "S1z": dict(judge="S1", cut=0.9995), "S5l": dict(judge="S5", cut=0.995), "S1l": dict(judge="S1", cut=0.998)}},
               decision="override none; extra-query cut = cut; min 3 inliers; Jetson: M = 5, context from 5, early exit at max(0.98, cut)",
               primary="S5z: 0 wrong and no more wrong than the base on 909; gain >= 0; interval above 0 = score gain, else safety only",
               robustness="inlier signals x0.5, 0.75, 1.25, 1.5, 2 on 909: wrong answers per scale")
    PRE_PATH.write_text(json.dumps(PRE, indent=1))
    print("WRITTEN", PRE_PATH.read_text()); print("sha256", hashlib.sha256(PRE_PATH.read_bytes()).hexdigest())
    raise SystemExit("pre-registration written; run again after the 909 dump to score")
PRE = json.loads(PRE_PATH.read_text()); print("pre-registration", PRE["written"], hashlib.sha256(PRE_PATH.read_bytes()).hexdigest())

pre-registration 2026-11-14T19:18:24 e1b696c5464d6a1fdb397ac9398d096526c10519255d34115d323e8f272ccf8d


In [2]:
DUMP = ROOT / "tests/Test_Iteration_10/results/rich_dump/nudt_fresh909"
assert len(list(DUMP.glob("photo_*.npz"))) == 300, "the 909 dump is not complete"
out = R / "signals" / "nudt909.csv.gz"
if not out.exists():
    build_signal_table(DUMP, "nudt909", 4).to_csv(out, index=False)
N909 = cj.add_context(cj.add_features(pd.read_csv(out)))
SAT = pd.read_csv(R / "signals" / "sat_train.csv.gz")
items = pd.read_csv(R / "sat_photos" / "sat_train" / "items.csv")
from sklearn.cluster import KMeans
fold = KMeans(5, random_state=0, n_init=10).fit_predict(items[["east", "north"]].to_numpy())
SAT["data"] = ["sat%d" % fold[p] for p in SAT.photo]; SAT_SETS = sorted(SAT.data.unique())
SAT = cj.add_context(cj.add_features(SAT))
F = cj.FEATURES_CTX; FR = [("rli" if f == "li" else "rln22" if f == "ln22" else f) for f in F]; AUG = [0.5, 0.75, 1.0, 1.5, 2.0]
def models(m):
    sat = SAT if m is None else cj.add_context(SAT, m=m)
    _, s1 = cj.judge_probability(sat, SAT_SETS, F, "gbm")
    lv = sr.far_levels(sat, m); prior = dict(far_inl=float(lv.far_inl.mean()), far_n22=float(lv.far_n22.mean()))
    s5, _, st = sr.fit_judge(sat, SAT_SETS, FR, {"rli", "z", "hull", "cover9"}, AUG, lambda x: sr.add_relative(x, prior, 30, m)); del st; gc.collect()
    return s1, s5, prior
SCALES = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]; RES, ROB = {}, []
for var, m in [("pc", None), ("jetson", 5)]:
    s1, s5, prior = models(m)
    for s in SCALES:
        t = sr.scale_signals(N909, s, m=m)
        jk = dict(jetson=True, m=5) if m else {}
        out_r = {"base": cj.baseline(t, 8, 11, min_choice=3) if var == "pc" else cj.baseline(t, 8, 8, jetson=True, m=5)}
        tr = sr.add_relative(t, prior, 30, m)
        P = {"S1": sr.predict(s1, t, F), "S5": sr.predict(s5, tr, FR)}
        for k, f in PRE["finalists"][var].items():
            ev = tr if f["judge"] == "S5" else t
            out_r[k] = cj.decide(ev, P[f["judge"]], f["cut"], wide_tau=f["cut"], override=None, min_choice=3,
                                 early_tau=max(0.98, f["cut"]) if m else None, **jk)
        if s == 1.0: RES[var] = (t, out_r)
        for k, v in out_r.items():
            sm = cj.summary(v).loc["nudt909"]
            ROB.append(dict(var=var, rule=k, scale=s, score=sm.score, wrong=int(sm.wrong), answered=sm.answered, queries=sm.queries))
ROB = pd.DataFrame(ROB); ROB.to_csv(R / "it2_909_scales.csv", index=False)
FRONT = bar_frontier(RES["pc"][0])
rows = []
for var, (t, r) in RES.items():
    for k, v in r.items():
        g = cj.paired_gain(r["base"], v, ["nudt909"])["nudt909"]; vf = versus_frontier(v, FRONT, var).iloc[0]; a = answer_change(r["base"], v).iloc[0]
        rows.append(dict(var=var, rule=k, score=vf.score, wrong=vf.wrong, answered=vf.n_answered, queries=vf.queries, gain=g, above_bar=vf.above_bar,
                         added_right=a.added_right, added_wrong=a.added_wrong, removed_wrong=a.removed_wrong))
        v.to_csv(R / f"it2_909_{var}_{k}.csv", index=False)
        save_manifest(R / f"it2_909_{var}_{k}.csv", manifest(var, cut=PRE["finalists"][var].get(k, {}).get("cut"), override=None, min_choice=3 if k != "base" else 1,
                      judge_features=None if k == "base" else PRE["finalists"][var][k]["judge"], judge_trained_on="satellite-made photos (T11)" if k != "base" else None,
                      data="nudt909", note="pre-registered T11"))
FINAL = pd.DataFrame(rows); FINAL.to_csv(R / "it2_909_final.csv", index=False); FINAL

,var,rule,score,wrong,answered,queries,gain,above_bar,added_right,added_wrong,removed_wrong
0,pc,base,54.24,2,279,8.100,"+0.00 [+0.00, +0.00]",0.00,0,0,0
1,pc,S5z,53.52,1,271,9.010,"-0.72 [-2.44, +1.08]",-0.28,1,1,2
2,pc,S1z,53.52,1,271,9.080,"-0.72 [-2.48, +1.08]",-0.28,2,1,2
3,pc,S5l,54.32,2,279,8.310,"+0.08 [-1.32, +1.48]",0.08,3,1,1
4,pc,S1l,54.72,2,281,8.310,"+0.48 [-0.92, +1.88]",1.28,4,1,1
5,jetson,base,52.72,4,276,6.370,"+0.00 [+0.00, +0.00]",0.00,0,0,0
6,jetson,S5z,53.28,1,271,2.110,"+0.56 [-0.92, +2.28]",0.48,1,0,3
7,jetson,S1z,52.88,0,266,2.360,"+0.16 [-1.56, +2.04]",0.08,0,0,4
8,jetson,S5l,53.96,2,277,1.937,"+1.24 [-0.12, +2.84]",1.24,4,0,2
9,jetson,S1l,53.28,2,274,2.100,"+0.56 [-0.72, +2.08]",0.76,2,0,2


In [3]:
pd.concat({"wrong": ROB.pivot_table(index=["var", "rule"], columns="scale", values="wrong").astype(int),
           "score": ROB.pivot_table(index=["var", "rule"], columns="scale", values="score").round(2)}, axis=1)

wrong                           score                                   
scale        0.50 0.75 1.00 1.25 1.50 2.00   0.50   0.75   1.00   1.25   1.50   2.00
var    rule                                                                         
jetson S1l      2    2    2    2    2    2  52.28  52.68  53.28  53.28  53.28  53.36
       S1z      0    0    0    0    0    0  51.56  52.56  52.88  52.88  53.08  53.28
       S5l      2    2    2    2    2    2  53.56  53.96  53.96  53.96  53.96  53.96
       S5z      1    0    1    1    1    1  52.68  53.76  53.28  53.28  53.28  53.28
       base     0    4    4    5    9   14  51.60  52.12  52.72  52.92  51.44  49.64
pc     S1l      2    2    2    2    2    2  54.12  54.72  54.72  54.92  54.92  54.92
       S1z      1    1    1    1    1    1  52.80  52.92  53.52  53.52  53.52  53.52
       S5l      2    2    2    2    2    2  54.12  54.52  54.32  54.32  54.32  54.32
       S5z      1    0    1    1    1    1  53.32  53.72  53.52  53.52  53.52  53.52
       base     0    1    2    6   11   17  51.60  53.44  54.24  52.64  50.64  48.24

In [4]:
pd.concat([failure_log(t, v, var).assign(var=var, rule=k) for var, (t, r) in RES.items() for k, v in r.items()]).groupby(["var", "rule"]).outcome.value_counts().unstack(fill_value=0)

outcome      answered_off  answered_right  answered_wrong  never_retrieved  retrieved_but_rejected  retrieved_but_unmatched
var    rule                                                                                                                
jetson S1l              0             272               2               11                       7                        8
       S1z              0             266               0               11                      14                        9
       S5l              0             275               2               11                       4                        8
       S5z              0             270               1               11                      10                        8
       base             0             272               4                9                       7                        8
pc     S1l              1             278               2                6                       8                        5
       S1z              0             270               1                7                      17                        5
       S5l              1             276               2                6                      10                        5
       S5z              0             270               1                7                      17                        5
       base             1             276               2                6                      10                        5